# HITL Approval

> Notebook generated from [`examples/subgraphs/09_hitl_approval.py`](../../examples/subgraphs/09_hitl_approval.py).

| Item | Value |
|------|-------|
| **Dataset** | AI Governance Decisions (embedded custom) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** proposal_writer → risk_assessor → approval_seed → interrupt() → hitl_gate → approve | reject | request_changes.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Human-in-the-Loop (HITL) Approval — Critical Decision Review Pipeline
=================================================================================

Dataset:  AI Governance Decisions (custom)
          5 AI deployment proposals at different risk levels
          (analogous to an ethics/safety committee approving production changes)

Pattern:  HITL with LangGraph interrupt() + Command(resume=...)
          • proposal_writer  → drafts the change proposal
          • risk_assessor    → assesses risk (LOW / MEDIUM / HIGH)
          • approval_seed    → seeds HITL metadata into state
          • human_approval   → interrupt() — pauses and waits for human decision
          • hitl_gate        → routes by action (approve / reject / request_changes)
          • finalizer        → executes the approved proposal
          • revision_handler → incorporates requested changes and resubmits

Modes:
  1. demo_interactive()   — interactive console loop (asks user for decision)
  2. demo_batch()         — simulates a sequence of automatic decisions
  3. demo_bypass()        — CI/CD mode: hitl_enabled=False → automatic approval
  4. demo_real_langgraph()— real LangGraph graph with MemorySaver and interrupt()

Usage:
  uv run python examples/subgraphs/09_hitl_approval.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import textwrap
from dataclasses import dataclass, field
from typing import Any

## Dataset — AI system change proposals

In [ ]:
@dataclass
class Proposal:
    id: str
    title: str
    description: str
    risk_level: str  # LOW | MEDIUM | HIGH
    risk_reasons: list[str]
    expected_impact: str
    rollback_plan: str


PROPOSALS: list[Proposal] = [
    Proposal(
        id="PROP-001",
        title="Enable GPT-4o model in production for customer support",
        description="Replace the current model (GPT-3.5) with GPT-4o in the support "
        "chatbot. Affects ~50,000 users/day.",
        risk_level="HIGH",
        risk_reasons=[
            "Change in response behavior not fully evaluated",
            "Higher cost per token (×3) may affect the budget",
            "No canary release planned — direct change to 100% of traffic",
        ],
        expected_impact="23% CSAT improvement; 18% reduction in escalations",
        rollback_plan="Revert MODEL_ID environment variable in < 5 min via feature flag",
    ),
    Proposal(
        id="PROP-002",
        title="Enable long-term memory for premium users",
        description="Store conversation summaries in persistent ChromaDB "
        "(PII included). Only opt-in users with explicit consent.",
        risk_level="HIGH",
        risk_reasons=[
            "Storing PII requires legal review (GDPR, LOPD)",
            "New attack vectors: memory extraction via prompt injection",
            "No retention policy defined yet",
        ],
        expected_impact="Context continuity across sessions; estimated NPS +12 pts",
        rollback_plan="Disable LONG_TERM_MEMORY_ENABLED flag; data wiped in 30 days",
    ),
    Proposal(
        id="PROP-003",
        title="Deploy autonomous PR review agent",
        description="Integrate the prismal code_review subgraph with GitHub Actions to "
        "review PRs automatically and merge if score >= 0.95.",
        risk_level="MEDIUM",
        risk_reasons=[
            "Auto-merge without human review in edge cases",
            "Possible false positive in security-critical code",
        ],
        expected_impact="40% reduction in review time; +30% bug detection",
        rollback_plan="Disable GitHub App; revert to manual review in < 2 min",
    ),
    Proposal(
        id="PROP-004",
        title="Enable RAG over internal knowledge base",
        description="Index 12,000 internal documents in ChromaDB and connect them "
        "to the assistant. Documents marked CONFIDENTIAL included.",
        risk_level="MEDIUM",
        risk_reasons=[
            "CONFIDENTIAL documents could leak in responses",
            "No role-based access control implemented yet",
        ],
        expected_impact="60% reduction in manual queries to the support team",
        rollback_plan="Remove ChromaDB collection; rollback in < 10 min",
    ),
    Proposal(
        id="PROP-005",
        title="Update safety prompt system to v2.1",
        description="Update SecurePromptBuilder and guardrails with new safety "
        "rules. Improves jailbreak detection from 71% to 89%.",
        risk_level="LOW",
        risk_reasons=[
            "Possible increase in false positives on legitimate queries (~2%)",
        ],
        expected_impact="Successful jailbreaks reduced from 29% to 11%",
        rollback_plan="Git revert of the prompt PR in < 1 min",
    ),
]

## Pipeline node simulation (without real LLM)

In [ ]:
RISK_COLORS = {"HIGH": "\033[91m", "MEDIUM": "\033[93m", "LOW": "\033[92m"}
RESET = "\033[0m"
BOLD = "\033[1m"
CYAN = "\033[96m"
DIM = "\033[2m"


def _risk_badge(level: str) -> str:
    color = RISK_COLORS.get(level, "")
    return f"{color}{BOLD}[{level}]{RESET}"


def proposal_writer(proposal: Proposal) -> dict[str, Any]:
    """Node 1 — Drafts the proposal artifact for review."""
    artifact = {
        "id": proposal.id,
        "title": proposal.title,
        "description": proposal.description,
        "expected_impact": proposal.expected_impact,
        "rollback_plan": proposal.rollback_plan,
        "status": "DRAFT",
    }
    print(f"\n{CYAN}{'─' * 64}{RESET}")
    print(f"{BOLD}[1/5] proposal_writer{RESET}  →  {proposal.id}: {proposal.title}")
    print(f"      {DIM}{textwrap.shorten(proposal.description, 72)}{RESET}")
    return artifact


def risk_assessor(proposal: Proposal, artifact: dict[str, Any]) -> dict[str, Any]:
    """Node 2 — Assesses risk and decides whether HITL is required."""
    risk = {
        "level": proposal.risk_level,
        "reasons": proposal.risk_reasons,
        "requires_hitl": proposal.risk_level in ("HIGH", "MEDIUM"),
    }
    badge = _risk_badge(proposal.risk_level)
    print(f"{BOLD}[2/5] risk_assessor{RESET}    →  Risk {badge}")
    for r in proposal.risk_reasons:
        print(f"        {DIM}• {r}{RESET}")
    return risk


def approval_seed(artifact_field: str, risk_level: str) -> dict[str, Any]:
    """Node 3 — Seeds HITL metadata into the state."""
    print(
        f"{BOLD}[3/5] approval_seed{RESET}    →  artifact_field={artifact_field!r}  risk={_risk_badge(risk_level)}"
    )
    return {
        "_hitl_artifact_field": artifact_field,
        "_hitl_risk_level": risk_level,
    }


def finalizer(proposal: Proposal, decision: str, modifications: dict[str, Any]) -> None:
    """Final node (approve path) — Executes the approved proposal."""
    title = modifications.get("title", proposal.title) or proposal.title
    print(f"\n{BOLD}[5/5] finalizer{RESET}        →  ✅  {BOLD}APPROVED and EXECUTED{RESET}")
    print(f"      Proposal: {title}")
    if modifications:
        print(f"      Modifications applied: {list(modifications.keys())}")
    print("      Final state: DEPLOYED")


def revision_handler(proposal: Proposal, modifications: dict[str, Any]) -> Proposal:
    """Reject/changes node — Incorporates feedback and updates the proposal."""
    print(f"\n{BOLD}[rev] revision_handler{RESET} →  🔄  Incorporating human feedback…")
    updated = Proposal(
        id=proposal.id,
        title=modifications.get("title", proposal.title),
        description=modifications.get("description", proposal.description),
        risk_level=proposal.risk_level,
        risk_reasons=proposal.risk_reasons,
        expected_impact=modifications.get("expected_impact", proposal.expected_impact),
        rollback_plan=modifications.get("rollback_plan", proposal.rollback_plan),
    )
    for k, v in modifications.items():
        print(f"      ← {k}: {v}")
    return updated

## HITL Simulator — interrupt() + Command(resume=...)

In [ ]:
@dataclass
class HitlInterrupt:
    """Represents the state suspended at an interrupt()."""

    artifact_field: str
    artifact: dict[str, Any]
    risk_level: str
    proposal: Proposal
    iteration: int = 0


@dataclass
class HitlDecision:
    action: str  # approve | reject | request_changes
    modifications: dict[str, Any] = field(default_factory=dict)
    feedback: str = ""


def _print_interrupt_payload(it: HitlInterrupt) -> None:
    """Display the interrupt payload to the human reviewer."""
    badge = _risk_badge(it.risk_level)
    print(f"\n{'═' * 64}")
    print(f"  ⏸  INTERRUPT — Approval required  {badge}  (iter {it.iteration + 1})")
    print(f"{'═' * 64}")
    print(f"  {BOLD}Proposal:{RESET} {it.artifact.get('id')} — {it.artifact.get('title')}")
    print(f"  {BOLD}Impact:{RESET}    {it.artifact.get('expected_impact')}")
    print(f"  {BOLD}Rollback:{RESET}  {it.artifact.get('rollback_plan')}")
    print(f"  {BOLD}Risk reasons:{RESET}")
    for r in it.proposal.risk_reasons:
        print(f"    • {r}")
    print(f"{'─' * 64}")
    print(
        f"  Valid options:  {BOLD}approve{RESET} | {BOLD}reject{RESET} | {BOLD}request_changes{RESET}"
    )
    print(f"{'─' * 64}")


def _human_input_interactive(it: HitlInterrupt) -> HitlDecision:
    """Read the human reviewer's decision from the console."""
    _print_interrupt_payload(it)
    while True:
        raw = input("  Decision [approve/reject/request_changes]: ").strip().lower()
        if raw in ("approve", "reject", "request_changes"):
            break
        print("  ⚠  Invalid option. Type: approve, reject, or request_changes")

    modifications = {}
    feedback = ""
    if raw == "request_changes":
        feedback = input("  Feedback (describe required changes): ").strip()
        rollback = input("  New rollback_plan (Enter to keep current): ").strip()
        if rollback:
            modifications["rollback_plan"] = rollback
    elif raw == "reject":
        feedback = input("  Rejection reason: ").strip()

    return HitlDecision(action=raw, modifications=modifications, feedback=feedback)


def _human_input_scripted(it: HitlInterrupt, script: list[HitlDecision]) -> HitlDecision:
    """Read the decision from a predefined script (for batch demo)."""
    idx = min(it.iteration, len(script) - 1)
    decision = script[idx]
    _print_interrupt_payload(it)
    badge_action = {
        "approve": f"\033[92m{BOLD}APPROVE{RESET}",
        "reject": f"\033[91m{BOLD}REJECT{RESET}",
        "request_changes": f"\033[93m{BOLD}REQUEST CHANGES{RESET}",
    }.get(decision.action, decision.action)
    print(f"\n  [scripted] Decision: {badge_action}")
    if decision.feedback:
        print(f"  [scripted] Feedback: {decision.feedback}")
    if decision.modifications:
        print(f"  [scripted] Modifications: {decision.modifications}")
    return decision

## Orchestrated pipeline (full simulation without LangGraph)

In [ ]:
MAX_ITERATIONS = 3


def run_hitl_pipeline(
    proposal: Proposal,
    decision_fn,  # callable(HitlInterrupt) -> HitlDecision
    bypass: bool = False,
) -> str:
    """
    Run the full HITL pipeline for a proposal.

    Flow:
      proposal_writer → risk_assessor
        ├── LOW  → finalizer (no HITL)
        └── MEDIUM/HIGH → approval_seed → [interrupt] → decision_fn()
              ├── approve           → finalizer
              ├── reject            → END (rejected)
              └── request_changes   → revision_handler → resubmit (max 3 iter)

    Args:
        proposal:    Proposal to evaluate.
        decision_fn: Callable that receives a HitlInterrupt and returns a HitlDecision.
        bypass:      If True, skip HITL (CI/CD mode).

    Returns:
        Final state: "approved" | "rejected" | "max_iterations"
    """
    iteration = 0
    current_proposal = proposal

    while iteration < MAX_ITERATIONS:
        # ── Node 1: proposal_writer ───────────────────────────────────────────
        artifact = proposal_writer(current_proposal)

        # ── Node 2: risk_assessor ─────────────────────────────────────────────
        risk = risk_assessor(current_proposal, artifact)

        # ── Configuration bypass (CI/CD mode) ─────────────────────────────────
        if bypass or not risk["requires_hitl"]:
            if bypass:
                print(
                    f"{BOLD}[hitl]{RESET}           →  ⚡ BYPASS (CI/CD mode) — automatic approval"
                )
            else:
                print(
                    f"{BOLD}[hitl]{RESET}           →  ✅ Risk LOW — no human approval required"
                )
            finalizer(current_proposal, "approve", {})
            return "approved"

        # ── Node 3: approval_seed ─────────────────────────────────────────────
        seed_meta = approval_seed(
            artifact_field="hitl_demo.proposal",
            risk_level=risk["level"],
        )

        # ── Node 4: human_approval (interrupt) ────────────────────────────────
        interrupt_state = HitlInterrupt(
            artifact_field=seed_meta["_hitl_artifact_field"],
            artifact={**artifact, "risk_reasons": risk["reasons"]},
            risk_level=seed_meta["_hitl_risk_level"],
            proposal=current_proposal,
            iteration=iteration,
        )

        print(f"{BOLD}[4/5] human_approval{RESET}   →  ⏸  interrupt() — waiting for decision…")

        # ── Command(resume=...) ───────────────────────────────────────────────
        decision = decision_fn(interrupt_state)

        # ── Node 5: hitl_gate ─────────────────────────────────────────────────
        print(
            f"\n{BOLD}[hitl_gate]{RESET}          →  action received: {BOLD}{decision.action}{RESET}"
        )

        if decision.action == "approve":
            finalizer(current_proposal, decision.action, decision.modifications)
            return "approved"

        if decision.action == "reject":
            print(f"\n{BOLD}[END]{RESET}              →  ❌  {BOLD}REJECTED{RESET}")
            if decision.feedback:
                print(f"      Reason: {decision.feedback}")
            return "rejected"

        if decision.action == "request_changes":
            current_proposal = revision_handler(current_proposal, decision.modifications)
            iteration += 1
            if iteration >= MAX_ITERATIONS:
                print(
                    f"\n{BOLD}[hitl_gate]{RESET}  →  ⚠  Maximum iterations ({MAX_ITERATIONS}) reached → automatically rejected"
                )
                return "max_iterations"
            print(
                f"\n  → Resubmitting revised proposal (iteration {iteration + 1}/{MAX_ITERATIONS})…"
            )
            continue

    return "max_iterations"

## DEMO 1 — Interactive (user decides in console)

In [ ]:
def demo_interactive() -> None:
    """Interactive pipeline: the user acts as the human reviewer."""
    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 1 — HITL Interactive{RESET}")
    print("  You act as a human reviewer of the AI Committee")
    print(f"{'═' * 64}")

    for i, proposal in enumerate(PROPOSALS[:3], 1):
        print(f"\n\n{'░' * 64}")
        print(f"  Proposal {i}/3  —  Risk {_risk_badge(proposal.risk_level)}")
        print(f"{'░' * 64}")
        result = run_hitl_pipeline(proposal, _human_input_interactive)
        print(f"\n  → Final result: {BOLD}{result.upper()}{RESET}")

    print(f"\n{'═' * 64}")
    print("  Interactive demo completed")
    print(f"{'═' * 64}")

## DEMO 2 — Batch (predefined automatic decisions)

In [ ]:
# Decision scenarios for each proposal
BATCH_SCRIPTS: dict[str, list[HitlDecision]] = {
    "PROP-001": [  # HIGH risk — requests changes first, then approves
        HitlDecision(
            action="request_changes",
            modifications={
                "rollback_plan": "Canary 5% → 25% → 100% with automatic rollback if error_rate > 1%"
            },
            feedback="Implement canary release before final approval",
        ),
        HitlDecision(action="approve"),
    ],
    "PROP-002": [  # HIGH risk — rejected outright
        HitlDecision(
            action="reject",
            feedback="Not allowed until completing PII legal audit (GDPR). Escalate to DPO.",
        ),
    ],
    "PROP-003": [  # MEDIUM risk — approved with modification
        HitlDecision(
            action="request_changes",
            modifications={
                "rollback_plan": "Disable auto-merge for files under security/; keep manual review there"
            },
            feedback="Exclude security/ directory from auto-merge",
        ),
        HitlDecision(action="approve"),
    ],
    "PROP-004": [  # MEDIUM risk — approved outright
        HitlDecision(action="approve"),
    ],
    "PROP-005": [  # LOW risk — no HITL required
        HitlDecision(action="approve"),  # never invoked
    ],
}


def demo_batch() -> None:
    """Pipeline with predefined decisions — shows all 5 scenarios."""
    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 2 — Batch (simulated decisions){RESET}")
    print("  Simulates a review committee with pre-programmed responses")
    print(f"{'═' * 64}")

    results: list[tuple[str, str, str]] = []

    for proposal in PROPOSALS:
        script = BATCH_SCRIPTS[proposal.id]

        def decision_fn(it, s=script):
            return _human_input_scripted(it, s)

        print(f"\n\n{'░' * 64}")
        print(f"  {proposal.id} — {proposal.title}")
        print(f"  Risk: {_risk_badge(proposal.risk_level)}")
        print(f"{'░' * 64}")

        result = run_hitl_pipeline(proposal, decision_fn)
        results.append((proposal.id, proposal.risk_level, result))
        print(f"\n  → Result: {BOLD}{result.upper()}{RESET}")

    # Summary
    print(f"\n\n{'═' * 64}")
    print(f"  {BOLD}Review Committee Summary{RESET}")
    print(f"{'═' * 64}")
    print(f"  {'ID':<12} {'Risk':<10} {'Result'}")
    print(f"  {'─' * 11} {'─' * 9} {'─' * 20}")
    for pid, risk, res in results:
        badge = _risk_badge(risk)
        result_fmt = (
            f"\033[92m{BOLD}{res.upper()}{RESET}"
            if res == "approved"
            else f"\033[91m{BOLD}{res.upper()}{RESET}"
        )
        print(f"  {pid:<12} {badge:<22} {result_fmt}")

    approved = sum(1 for _, _, r in results if r == "approved")
    print(f"\n  Approved: {approved}/{len(results)}")
    print(f"{'═' * 64}")

## DEMO 3 — CI/CD Bypass

In [ ]:
def demo_bypass() -> None:
    """Show CI/CD behavior: hitl_enabled=False → automatic approval."""
    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 3 — CI/CD Bypass (hitl_enabled=False){RESET}")
    print("  All proposals are automatically approved without interrupt")
    print(f"{'═' * 64}")

    for proposal in PROPOSALS:
        print(f"\n  {proposal.id} — Risk {_risk_badge(proposal.risk_level)}")
        result = run_hitl_pipeline(
            proposal,
            decision_fn=lambda _: HitlDecision(action="approve"),  # never called
            bypass=True,
        )
        print(f"  Result: {BOLD}{result}{RESET}")

    print("\n  → All proposals processed without human intervention.")
    print(f"{'═' * 64}")

## DEMO 4 — Real LangGraph with MemorySaver and interrupt()

In [ ]:
async def demo_real_langgraph() -> None:
    """
    Real LangGraph graph using MemorySaver and the native interrupt().

    Shows the exact pattern used by prismal in production:
      graph.invoke(state, config)           → suspends at interrupt()
      graph.invoke(Command(resume=...), config) → resumes with human decision
    """
    try:
        from langgraph.checkpoint.memory import MemorySaver
        from langgraph.graph import END, StateGraph
        from langgraph.graph.message import MessagesState
        from langgraph.types import Command, interrupt

        from prismal.agents.subgraphs.gates import (
            hitl_gate,
            human_approval_node,
            seed_hitl_metadata,
        )
    except ImportError as e:
        print(f"\n  ⚠  Dependency not available: {e}")
        print("     Install with: uv pip install -e '.[dev,all]'")
        return

    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 4 — Real LangGraph with MemorySaver + interrupt(){RESET}")
    print(f"{'═' * 64}")

    proposal = PROPOSALS[0]  # PROP-001 (HIGH risk)

    # ── Define the graph state ────────────────────────────────────────────────
    from typing import TypedDict

    class ReviewState(TypedDict, total=False):
        proposal_id: str
        proposal_title: str
        risk_level: str
        artifact: dict
        metadata: dict
        result: str

    # ── Graph nodes ───────────────────────────────────────────────────────────
    def _writer_node(state: ReviewState) -> ReviewState:
        print(f"\n  [writer]   Drafting proposal {state['proposal_id']}…")
        artifact = {
            "id": state["proposal_id"],
            "title": state["proposal_title"],
            "description": "GPT-4o production deployment proposal",
            "rollback_plan": "Revert feature flag in < 5 min",
        }
        return {"artifact": artifact}

    def _risk_node(state: ReviewState) -> ReviewState:
        print(f"  [risk]     Risk level: {state['risk_level']}")
        meta = state.get("metadata", {})
        meta["proposal_risk"] = state["risk_level"]
        return {"metadata": meta}

    # seed_hitl_metadata returns a node function
    _seed_node = seed_hitl_metadata(
        artifact_field="artifact",
        risk_level="HIGH",
    )

    def _finalizer_node(state: ReviewState) -> ReviewState:
        meta = state.get("metadata", {})
        action = meta.get("_hitl_last_action", "approved (bypass)")
        print(f"  [finalizer] ✅ Proposal executed. HITL action: {action}")
        return {"result": "DEPLOYED"}

    def _rejected_node(state: ReviewState) -> ReviewState:
        print("  [rejected]  ❌ Proposal rejected.")
        return {"result": "REJECTED"}

    # ── Build graph ───────────────────────────────────────────────────────────
    _gate = hitl_gate(
        artifact_field="artifact",
        on_approve="finalizer",
        on_reject="rejected",
        risk_level="HIGH",
    )

    builder = StateGraph(ReviewState)
    builder.add_node("writer", _writer_node)
    builder.add_node("risk", _risk_node)
    builder.add_node("approval_seed", _seed_node)
    builder.add_node("human_approval", human_approval_node)
    builder.add_node("finalizer", _finalizer_node)
    builder.add_node("rejected", _rejected_node)

    builder.set_entry_point("writer")
    builder.add_edge("writer", "risk")
    builder.add_edge("risk", "approval_seed")
    builder.add_edge("approval_seed", "human_approval")
    builder.add_conditional_edges("human_approval", _gate)
    builder.add_edge("finalizer", END)
    builder.add_edge("rejected", END)

    checkpointer = MemorySaver()
    graph = builder.compile(checkpointer=checkpointer, interrupt_before=["human_approval"])

    # ── First invocation — suspends at human_approval ─────────────────────────
    config = {"configurable": {"thread_id": "demo-hitl-001"}}
    initial_state: ReviewState = {
        "proposal_id": proposal.id,
        "proposal_title": proposal.title,
        "risk_level": proposal.risk_level,
        "artifact": {},
        "metadata": {},
        "result": "",
    }

    print(f"\n  Invoking graph for {proposal.id}…")
    graph.invoke(initial_state, config)

    # Detect that we are at the interrupt
    state_snapshot = graph.get_state(config)
    pending = state_snapshot.next
    print(f"\n  ⏸  Graph suspended. Next node: {pending}")
    print(f"  Artifact in state: {state_snapshot.values.get('artifact', {}).get('title', '?')}")

    # ── Simulate human decision 1: request_changes ────────────────────────────
    print("\n  [human] Decision: request_changes — add canary release")
    decision_1 = {
        "action": "request_changes",
        "modifications": {"rollback_plan": "Canary 5%→25%→100% with auto-rollback"},
    }
    graph.invoke(Command(resume=decision_1), config)

    # The graph reaches rejected (on_reject) because request_changes → on_reject
    # In this simplified graph, on_reject = "rejected". In the real dev_pipeline
    # on_reject = "developer" to regenerate.
    state2 = graph.get_state(config)
    print(f"\n  State after request_changes: {state2.values.get('result', 'pending')}")
    print(f"  Metadata HITL: action={state2.values.get('metadata', {}).get('_hitl_last_action')}")

    # ── Second execution: approve ─────────────────────────────────────────────
    config2 = {"configurable": {"thread_id": "demo-hitl-002"}}
    print("\n  ─── Second execution (different thread) → approve ───")
    graph.invoke(initial_state, config2)
    print("\n  [human] Decision: approve")
    graph.invoke(Command(resume={"action": "approve"}), config2)
    state_final = graph.get_state(config2)
    print(f"\n  Final state: {state_final.values.get('result')}")

    print(f"\n{'═' * 64}")
    print("  Real LangGraph demo completed.")
    print("  Key pattern:")
    print("    graph.invoke(state, config)              → suspends")
    print("    graph.invoke(Command(resume=...), config) → resumes")
    print(f"{'═' * 64}")

## HITL pattern explanation

In [ ]:
def print_pattern_explanation() -> None:
    """Print the pattern diagram and the usage guide."""
    print(f"""
{"═" * 64}
  {BOLD}HITL Pattern — Human-in-the-Loop{RESET}
{"═" * 64}

  {BOLD}Full flow:{RESET}

    proposal_writer
         │
    risk_assessor
         │
    ┌────┴────────────────────────────────┐
    │ risk=LOW                 risk=MEDIUM/HIGH
    │                               │
    │                        approval_seed
    │                        (seeds metadata)
    │                               │
    │                        human_approval
    │                        interrupt() ⏸
    │                               │
    │                          decision
    │                      ┌─────┬──┴────────────┐
    │                   approve  reject  request_changes
    │                      │       │           │
    finalizer          finalizer END    revision_handler
    (bypass)           (deploy)          (loop ≤ 3 iter)

  {BOLD}Framework API:{RESET}

    from prismal.agents.subgraphs.gates import (
        seed_hitl_metadata,    # node that seeds _hitl_artifact_field
        human_approval_node,   # async node with interrupt()
        hitl_gate,             # conditional edge (approve/reject/changes)
    )

    # Wiring in StateGraph:
    builder.add_node("approval_seed",  seed_hitl_metadata("my_artifact", "HIGH"))
    builder.add_node("human_approval", human_approval_node)
    builder.add_conditional_edges("human_approval",
        hitl_gate("my_artifact", on_approve="finalizer", on_reject="generator"))

  {BOLD}LangGraph lifecycle:{RESET}

    # Invocation 1 — suspends at interrupt_before=["human_approval"]
    graph.invoke(state, config)

    # The human reviewer inspects the state and decides:
    decision = {{"action": "approve"}}                    # or:
    decision = {{"action": "request_changes",
                "modifications": {{"rollback_plan": "..."}}}}

    # Invocation 2 — resumes from the checkpoint
    graph.invoke(Command(resume=decision), config)

  {BOLD}Bypass CI/CD:{RESET}

    # In .env or settings:
    PRISMAL_HITL_ENABLED=false   # → automatic approval

    # Or by code (tests):
    hitl_gate(..., bypass_condition=lambda _: True)

{"═" * 64}""")

## main

In [ ]:
async def main() -> None:
    print(f"""
{"═" * 64}
  {BOLD}HITL Approval — Human Review Patterns in AI Agents{RESET}
  Dataset: AI Governance Decisions (5 deployment proposals)
  Framework: prismal / LangGraph interrupt()
{"═" * 64}
  Available modes:
    1. Interactive — you are the reviewer (console)
    2. Batch       — simulated decisions (automatic demo)
    3. Bypass      — CI/CD without human intervention
    4. LangGraph   — real graph with MemorySaver + interrupt()
    5. All         — runs demos 2, 3 and 4 in sequence
    6. Pattern     — shows HITL pattern explanation
{"─" * 64}""")

    choice = input("  Select mode [1-6] (Enter = 2): ").strip() or "2"

    if choice == "1":
        demo_interactive()
    elif choice == "2":
        demo_batch()
    elif choice == "3":
        demo_bypass()
    elif choice == "4":
        await demo_real_langgraph()
    elif choice == "5":
        demo_batch()
        demo_bypass()
        await demo_real_langgraph()
    elif choice == "6":
        print_pattern_explanation()
    else:
        print("  Invalid option. Running default batch demo.")
        demo_batch()

    print(f"\n  Tip: {DIM}hitl_enabled=True  → human review required")
    print(f"       hitl_enabled=False → automatic bypass (CI/CD mode){RESET}\n")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()